In [ ]:
"""
Distal-rate HMM demo that reproduces "function word appears/disappears" with identical acoustics.
- Fast context: local prior bump near expected onset -> inserts "or"
- Slow context: local penalty (+ onset delay) -> skips "or"
"""
import numpy as np
from collections import Counter
from scipy.stats import norm, gamma as gamma_dist

np.random.seed(123)

In [ ]:
# --- Model Parameters ---
# Labels for the latent “word” states in the toy sentence
z_true = ["lei", "sure", "or", "time"]


# Make 'or' acoustically indistinguishable from 'sure' to force reliance on the prior
# so the decoder must rely on the prior
# This reproduces the acoustically blended weak onset
mu_true = {
    "lei": np.array([-1.0]),
    "sure": np.array([0.1]),
    "or":   np.array([0.1]),   # indistinguishable from 'sure'
    "time": np.array([1.0])
}

sigma_true = {s: 0.30 for s in z_true}  # variance (std ≈ 0.5477)


# Prior hyperparameters for per-state “duration scale” γ
# We don’t emphasize Bayesian learning here, we use γ to make stickiness depend on rate
hyperparams = {
    'gamma': {'alpha': 2.0, 'beta': 5.0},
}


# RLE compress states (['lei','lei','lei','sure',...] → ['lei','sure',...]) for readable outputs
def compress_sequence(seq):
    out = []
    for s in seq:
        if not out or out[-1] != s:
            out.append(s)
    return out


# Core HMM with distal-rate prior as a time-local bias

class RateBiasedHMM:
    """
    Strong stickiness version (gives very clean split: FAST 100% 'or', SLOW 0% 'or').
    """
    # This version makes states persistent. It’s intentionally strong so we see the effect cleanly
    def __init__(self, states, hyperparams):
        self.states = states
        self.state_map = {name: i for i, name in enumerate(states)}
        self.idx_to_state = {i: name for name, i in self.state_map.items()}
        self.n_states = len(states)
        self.gamma = {s: gamma_dist.rvs(a=hyperparams['gamma']['alpha'],
                                        scale=1/hyperparams['gamma']['beta'])
                      for s in states}
        self.pi_bar = self._pi_bar_forward_skip()

    # Draw γ per state from Gamma (just to vary stickiness across states a bit).
    # pi_bar is the base forward skeleton (left→right) with a possible skip from sure→time.
    # The skip is the mechanism by which “or” can disappear.

    def _pi_bar_forward_skip(self):
        """Strict forward transitions; allow sure->time skip; no leakage elsewhere."""
        # Strict forward transitions: no leakage or loops, just the natural flow of the phrase.
        # skip path sure→time = “we didn’t perceive the function word.”
        n = self.n_states
        T = np.zeros((n, n))
        i_lei, i_sure, i_or, i_time = [self.state_map[s] for s in ["lei","sure","or","time"]]
        T[i_lei, i_sure] = 1.0                  # lei -> sure
        T[i_sure, i_or] = 0.5; T[i_sure, i_time] = 0.5  # sure -> or OR time
        T[i_or, i_time] = 1.0                   # or -> time
        T[i_time, i_time] = 1.0                 # time -> time
        return T


    @staticmethod
    # Converts γ + rate r into self-transition probability κ (“stickiness”).
    # Slower distal rate ⇒ higher stickiness (you expect longer chunks; you “hold” the current state longer).
    # rate shapes timing expectations before the onset.
    def _stickiness_map(gamma, r):
        # Strong stickiness; slow -> ~0.98, fast -> ~0.90

        base = 1.8
        val = 1.0 - np.exp(-base * (gamma + 0.3) / (r + 1e-6))
        return float(np.clip(val, 0.80, 0.995))


    # Builds a time-varying transition matrix over the sequence.
    # Around time t* (the expected onset for “or”), we locally bump the transition probability sure→or (if fast context) or penalize it (if slow context)
    def _time_varying_T_log(self, r, T_len, t_star, window=4, bias_fast=3.0, bias_slow=-4.0, tstar_shift=0):
        bias = bias_fast if r >= 6.0 else bias_slow
        log_T_seq = []
        i_sure = self.state_map["sure"]
        j_or = self.state_map["or"]
        for t in range(T_len):
            T = np.zeros((self.n_states, self.n_states))
            for i, s_from in enumerate(self.states):
                kappa = self._stickiness_map(self.gamma[s_from], r)
                row = (1 - kappa) * self.pi_bar[i].copy()
                row[i] += kappa
                T[i, :] = row
            if abs(t - (t_star + tstar_shift)) <= window:
                T[i_sure, j_or] *= np.exp(bias)
            T = T / T.sum(axis=1, keepdims=True)
            log_T_seq.append(np.log(T + 1e-12))
        return log_T_seq



    # Likelihood over states given the fixed acoustics
    #  "or" and "sure" are acoustically the same
    def emission_log_probs(self, a_data):
        T = len(a_data)
        logp = np.zeros((T, self.n_states))
        for t, x in enumerate(a_data):
            for i, s in enumerate(self.states):
                logp[t, i] = norm.logpdf(x, mu_true[s], np.sqrt(sigma_true[s]))
        return logp
    # Standard dynamic programming, but using the time-local biased transitions.
    # Start at "lei". The path will insert "or" under fast context (bump), and skip it under slow (penalty + delay).
    def viterbi(self, a_data, r, t_star, window=4, bias_fast=3.0, bias_slow=-4.0, tstar_shift=0):
        log_T_seq = self._time_varying_T_log(r, len(a_data), t_star, window, bias_fast, bias_slow, tstar_shift)
        logE = self.emission_log_probs(a_data)
        T_len = len(a_data)
        V = np.full((T_len, self.n_states), -np.inf)
        B = np.zeros((T_len, self.n_states), dtype=int)
        V[0, self.state_map["lei"]] = logE[0, self.state_map["lei"]]
        for t in range(1, T_len):
            prev = V[t-1][:, None] + log_T_seq[t]
            B[t] = np.argmax(prev, axis=0)
            V[t] = np.max(prev, axis=0) + logE[t]
        path = np.zeros(T_len, dtype=int)
        path[-1] = np.argmax(V[-1])
        for t in range(T_len-2, -1, -1):
            path[t] = B[t+1, path[t+1]]
        return [self.idx_to_state[i] for i in path]





# Same idea but with weaker stickiness, so emissions and the local bias compete more smoothly.
class RateBiasedHMM_Softer(RateBiasedHMM):
    """
    Softer stickiness version (lets acoustics + prior compete).
    """
    @staticmethod
    def _stickiness_map(gamma, r):
        base = 0.9  # much lower => kappa ~0.50-0.90
        val = 1.0 - np.exp(-base * (gamma + 0.3) / (r + 1e-6))
        return float(np.clip(val, 0.50, 0.90))

# --- Build identical target acoustic segment for both conditions ---

# Lengths by region so we can place t* inside "sure", i.e., where the blended weak onset sits
L_lei, L_sure, L_or, L_time = 30, 35, 12, 25
target_states = ["lei"]*L_lei + ["sure"]*L_sure + ["or"]*L_or + ["time"]*L_time
T_len = len(target_states)
t_star_base = L_lei + L_sure//2  # expected 'or' onset inside 'sure'


# Critical control: the exact same samples are decoded twice (fast vs slow)
def sample_target_acoustics():
    # Pair-constant acoustics; 'or' and 'sure' sampled from same distribution
    arr = []
    for s in target_states:
        arr.append(np.random.normal(mu_true[s], np.sqrt(sigma_true[s])))
    return np.array(arr).reshape(-1,1)


# Fast: local bump near t* → encourages sure→or at that time.
# Slow: local penalty and later expected onset → encourages sure→time
def run_paired(hmm, a, r_slow=4.0, r_fast=7.0, window=4, bias_fast=3.0, bias_slow=-4.0, tstar_shift_slow=8):
    # Fast: bump at t_star; Slow: penalize and delay expectation
    seq_fast = hmm.viterbi(a, r_fast, t_star=t_star_base, window=window,
                           bias_fast=bias_fast, bias_slow=bias_slow, tstar_shift=0)
    seq_slow = hmm.viterbi(a, r_slow, t_star=t_star_base, window=window,
                           bias_fast=bias_fast, bias_slow=bias_slow, tstar_shift=tstar_shift_slow)
    return compress_sequence(seq_fast), compress_sequence(seq_slow)



# Monte Carlo simulations
# Runs many paired trials where acoustics are re-sampled but shared within a pair.
# Reports how often “or” appears in fast vs slow.
def run_sim(n_trials=400, model="strong", r_slow=4.0, r_fast=7.0, window=4,
            bias_fast=3.0, bias_slow=-4.0, tstar_shift_slow=8):
    H = RateBiasedHMM if model=="strong" else RateBiasedHMM_Softer
    hmm = H(z_true, hyperparams)
    fast_res, slow_res = [], []
    for _ in range(n_trials):
        a = sample_target_acoustics()   # identical acoustics per pair
        f, s = run_paired(hmm, a, r_slow, r_fast, window, bias_fast, bias_slow, tstar_shift_slow)
        fast_res.append(tuple(f))
        slow_res.append(tuple(s))

    def summarize(label, counter):
        total = sum(counter.values())
        print(f"\n{label} (N={total})")
        for seq, cnt in Counter(counter).most_common(6):
            print(f"  {list(seq)!s:<28} | {cnt:4d} | {100*cnt/total:5.1f}%")

    c_fast = Counter(fast_res)
    c_slow = Counter(slow_res)
    summarize(f"FAST r={r_fast}", c_fast)
    summarize(f"SLOW r={r_slow}", c_slow)

    def has_or(seq): return "or" in seq
    fast_or = sum(1 for x in fast_res if has_or(x))
    slow_or = sum(1 for x in slow_res if has_or(x))
    print("\nAppearance of 'or':")
    print(f"  FAST: {fast_or}/{n_trials} = {100*fast_or/n_trials:4.1f}%")
    print(f"  SLOW: {slow_or}/{n_trials} = {100*slow_or/n_trials:4.1f}%")

if __name__ == "__main__":
    # Example 1: very clean split (FAST 100% 'or', SLOW 0% 'or')
    print("=== Strong split demo ===")
    run_sim(n_trials=400, model="strong",
            bias_fast=3.0, bias_slow=-4.2, window=4, tstar_shift_slow=8)

    # Example 2: softer split (tune to taste)
    print("\n=== Softer split demo ===")
    run_sim(n_trials=300, model="softer",
            bias_fast=0.8, bias_slow=-1.0, window=3, tstar_shift_slow=6)
